In [ ]:
!pip install transformers peft accelerate bitsandbytes fastapi uvicorn pyngrok nest_asyncio

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

ADAPTER_PATH = "AbdulazizCS/logpulse-qwen-classifier"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

import torch
import json
import re

app = FastAPI()


# ── SYSTEM_PROMPT must match the training prompt verbatim ──────────────────
# Source: LogPulse_Qwen_clean.ipynb — "Define SYSTEM_PROMPT" cell.
# The loose serving prompt that was here before ("category": "string") caused
# training-serving skew: the model emitted variants like "DatabaseError"
# instead of the canonical "Database" it was trained to produce.
# ─────────────────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a DevOps log classifier. Given a raw log line, output ONLY a valid JSON object with these exact fields:

{
  "category": one of ["Network", "Database", "Memory", "Security", "Application", "System"],
  "source": one of ["Docker", "Kubernetes", "Server", "Database", "OS", "Network_Device"],
  "severity": one of ["Warning", "Critical", "Fatal"],
  "summary": a short human-readable summary of the problem (max 15 words)
}

FIELD DEFINITIONS:
- "category" = WHAT kind of issue it is:
    • Network    → connections, timeouts, packets, sockets, DNS, ports
    • Database   → data storage, blocks, reads/writes, queries, replication, file system data ops (e.g. HDFS block storage)
    • Memory     → RAM, heap, OOM, leaks, allocation, cache eviction
    • Security   → auth, login attempts, permissions, intrusion, SSH, certificates
    • Application→ app-level logic, services, request handling, normal app operations
    • System     → OS/kernel/hardware level, daemons, processes, system services

- "source" = WHERE the log came from:
    • Docker        → container runtime logs
    • Kubernetes    → pods, deployments, kubelet, scheduler
    • Server        → application server processes (e.g. HDFS DataNode/NameNode)
    • Database      → dedicated DB engines
    • OS            → operating system / kernel
    • Network_Device→ routers, switches, firewalls

DISAMBIGUATION RULES (read carefully):
- HDFS / distributed-storage logs about BLOCKS, DataNode, NameNode storage operations → category "Database", source "Server".
- SSH / login / authentication logs → category "Security", source "OS".
- Kernel / hardware / daemon / kernel-panic logs → category "System", source "OS".
- Container crash / image pull / pod / OOMKilled logs → source "Kubernetes" or "Docker"; pick category by the actual problem (OOMKilled → "Memory", network timeout → "Network", crash loop → "Application").
- Be CONSISTENT: the same kind of log must always get the same category. Do not switch between "Application" and "System" for identical log types.

SEVERITY RULES:
- INFO / normal operational logs → "Warning"
- ERROR / failed operations / exceptions → "Critical"
- FATAL / crash / data loss / system down → "Fatal"

OUTPUT RULES:
- Output ONLY the JSON object. No markdown, no backticks, no explanation.
- Always pick the single best-fitting value for each field.

EXAMPLES:
Log: "081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049 terminating"
Output: {"category": "Database", "source": "Server", "severity": "Warning", "summary": "HDFS DataNode packet responder terminating for a storage block"}

Log: "Dec 10 06:55:46 LabSZ sshd[24200]: Failed password for invalid user webmaster from 173.234.31.186 port 38926 ssh2"
Output: {"category": "Security", "source": "OS", "severity": "Critical", "summary": "Failed SSH login attempt for invalid user from external IP"}

Log: "2026-05-20T10:14:22Z pod/api-server-7d9 OOMKilled: container exceeded memory limit 512Mi"
Output: {"category": "Memory", "source": "Kubernetes", "severity": "Critical", "summary": "Kubernetes pod killed for exceeding its memory limit"}

Log: "kernel: [12345.67] CPU0: Core temperature above threshold, cpu clock throttled"
Output: {"category": "System", "source": "OS", "severity": "Warning", "summary": "CPU core temperature exceeded threshold, clock throttled"}"""


class PromptRequest(BaseModel):
    prompt: str


@app.post("/generate")
async def generate(data: PromptRequest):

    try:

        # Build the conversation the same way the model was trained:
        # system = exact training SYSTEM_PROMPT, user = the raw log line.
        # apply_chat_template adds Qwen's special tokens and the chat structure
        # the model expects; add_generation_prompt=True primes it to produce
        # the assistant (JSON) response — identical to evaluation inference.
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": data.prompt},
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():

            outputs = model.generate(
                input_ids=inputs,
                max_new_tokens=128,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode only the newly generated tokens (skip the prompt tokens)
        generated_tokens = outputs[0][inputs.shape[1]:]

        output_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
        )

        match = re.search(r"\{.*\}", output_text, re.DOTALL)

        if match:

            cleaned_json = match.group(0)

            try:

                parsed = json.loads(cleaned_json)

                return parsed

            except Exception:

                return {
                    "error": "Invalid JSON generated",
                    "raw_output": output_text,
                }

        return {
            "error": "No JSON found",
            "raw_output": output_text,
        }

    except Exception as e:

        print("ERROR:", str(e))

        return {
            "error": str(e),
        }


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from threading import Thread

nest_asyncio.apply()

PORT = 8000

public_url = ngrok.connect(PORT)
print("=" * 50)
print(f"🚀 ✅ Your API URL is: {public_url.public_url}/generate")
print("=" * 50)


def run_app():
    uvicorn.run(app, host="0.0.0.0", port=PORT)

thread = Thread(target=run_app)
thread.start()

🚀 ✅ Your API URL is: https://graded-regalia-unused.ngrok-free.dev/generate


INFO:     Started server process [12251]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
